# 📈 Revenue & COGS Forecasting — XGBoost + Recursive Prediction

**Approach:** Dual XGBoost models (Revenue & COGS) trained on engineered features with projection-based KPIs, recursive forecasting from 2023–2024.

| Stage | Details |
|---|---|
| Train period | 2012–2022 inclusive (recent years weighted ×5) |
| Validation | In-sample check on 2022 |
| Forecast horizon | 2023-01-01 → 2024-07-01 |

## 1 · Setup — Libraries & Config

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ── Config ───────────────────────────────────────────────────────────────────
BASE     = '/kaggle/input/competitions/datathon-2026-round-1'
RECENT_W = 5.0

FEATURES = [
    'trend_day', 'month', 'is_peak', 'is_bottom',
    'is_wed_peak', 'is_sat_trough', 'dow_sin',
    'proj_CR', 'proj_AOV', 'proj_retention_rate',
    'Revenue_lag7', 'Revenue_lag1', 'Revenue_roll7',
    'sessions_lag7', 'total_orders_lag7'
]
PROJ_COLS = ['CR', 'AOV', 'retention_rate']

## 2 · Data Loading

In [ ]:
# ── Load raw files ─────────────────────────────────────────────────────────
sales   = pd.read_csv(f'{BASE}/sales.csv',      parse_dates=['Date']).sort_values('Date')
orders  = pd.read_csv(f'{BASE}/orders.csv',      parse_dates=['order_date']).rename(columns={'order_date': 'Date'})
traffic = pd.read_csv(f'{BASE}/web_traffic.csv', parse_dates=['date']).rename(columns={'date': 'Date'})

# ── Orders: tag new vs repeat customers ───────────────────────────────────
orders['first_date'] = orders.groupby('customer_id')['Date'].transform('min')
orders['is_new']     = (orders['Date'] == orders['first_date']).astype(int)
orders['is_repeat']  = (orders['Date'] >  orders['first_date']).astype(int)

orders_daily = (
    orders.groupby('Date')
          .agg(total_orders=('order_id', 'nunique'),
               is_new=('is_new', 'sum'),
               is_repeat=('is_repeat', 'sum'))
          .reset_index()
)

# ── Merge into one daily dataframe ────────────────────────────────────────
df = (
    sales
    .merge(orders_daily,                on='Date', how='left')
    .merge(traffic[['Date','sessions']], on='Date', how='left')
    .fillna(0)
)

print(f"Raw data: {len(df):,} rows | {df['Date'].min().date()} → {df['Date'].max().date()}")
df.head()

## 3 · Feature Engineering

In [ ]:
# ── KPI features ──────────────────────────────────────────────────────────
df['AOV']            = df['Revenue'] / (df['total_orders'] + 1)
df['CR']             = df['total_orders'] / (df['sessions'] + 1)
df['retention_rate'] = df['is_repeat'] / (df['is_new'] + df['is_repeat'] + 1)
df['trend_day']      = (df['Date'] - df['Date'].min()).dt.days

# ── Linear projection for KPIs (fitted on 2019+) ──────────────────────────
projection_models = {}
for col in PROJ_COLS:
    mask = df['Date'].dt.year >= 2019
    lr   = LinearRegression().fit(df.loc[mask, ['trend_day']], df.loc[mask, col])
    df[f'proj_{col}'] = lr.predict(df[['trend_day']])
    projection_models[col] = lr

# ── Calendar features ─────────────────────────────────────────────────────
df['month']         = df['Date'].dt.month
df['day_of_week']   = df['Date'].dt.dayofweek
df['is_peak']       = df['month'].isin([3, 4, 5, 6]).astype(int)
df['is_bottom']     = df['month'].isin([10, 11, 12, 1]).astype(int)
df['is_wed_peak']   = (df['day_of_week'] == 2).astype(int)
df['is_sat_trough'] = (df['day_of_week'] == 5).astype(int)
df['dow_sin']       = np.sin(2 * np.pi * df['day_of_week'] / 7)

# ── Lag / rolling features ────────────────────────────────────────────────
df['Revenue_lag1']      = df['Revenue'].shift(1)
df['Revenue_lag7']      = df['Revenue'].shift(7)
df['Revenue_roll7']     = df['Revenue'].shift(1).rolling(7).mean()
df['COGS_lag1']         = df['COGS'].shift(1)
df['COGS_lag7']         = df['COGS'].shift(7)
df['sessions_lag7']     = df['sessions'].shift(7)
df['total_orders_lag7'] = df['total_orders'].shift(7)

df = df.dropna().reset_index(drop=True)
print(f"After feature engineering: {len(df):,} rows | {df['Date'].min().date()} → {df['Date'].max().date()}")

## 4 · Model Training — Revenue & COGS

> **Note:** Training includes 2022 (`year ≤ 2022`) to capture the most recent seasonal patterns before forecasting 2023–2024.

In [ ]:
XGB_PARAMS = dict(
    n_estimators=1500, learning_rate=0.01, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, nthread=1
)

# ── Train / weight mask ──────────────────────────────────────────────────────
train_mask = df['Date'].dt.year <= 2022
X_train    = df.loc[train_mask, FEATURES]
weights    = np.where(df.loc[train_mask, 'Date'].dt.year >= 2019, RECENT_W, 1.0)

model_rev  = XGBRegressor(**XGB_PARAMS)
model_rev.fit(X_train, np.log1p(df.loc[train_mask, 'Revenue']), sample_weight=weights)

model_cogs = XGBRegressor(**XGB_PARAMS)
model_cogs.fit(X_train, np.log1p(df.loc[train_mask, 'COGS']), sample_weight=weights)

print('✅ Both models trained successfully.')
print(f'   Train rows: {train_mask.sum():,} | {df.loc[train_mask, "Date"].min().date()} → {df.loc[train_mask, "Date"].max().date()}')

## 5 · Validation — In-sample Check 2022

In [ ]:
val_mask     = df['Date'].dt.year == 2022
y_val_actual = df.loc[val_mask, 'Revenue']
y_pred_val   = np.expm1(model_rev.predict(df.loc[val_mask, FEATURES]))

rmse = np.sqrt(mean_squared_error(y_val_actual, y_pred_val))
mae  = mean_absolute_error(y_val_actual, y_pred_val)
r2   = r2_score(y_val_actual, y_pred_val)

print('── IN-SAMPLE CHECK (REVENUE 2022) ──')
print(f'  RMSE : {rmse:>12,.0f}')
print(f'  MAE  : {mae:>12,.0f}')
print(f'  R²   : {r2:>12.4f}')

### 5.1 · Actual vs Predicted — 2022

In [ ]:
comparison_df = pd.DataFrame({
    'Date':    df.loc[val_mask, 'Date'].values,
    'Actual':  y_val_actual.values,
    'Predict': y_pred_val
}).sort_values('Date')

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(comparison_df['Date'], comparison_df['Actual'],  label='Actual Revenue',    color='#1f77b4', alpha=0.8, linewidth=2)
ax.plot(comparison_df['Date'], comparison_df['Predict'], label='Predicted Revenue', color='#ff7f0e', linestyle='--', linewidth=2)
ax.set_title(f'Actual vs Predicted Revenue — 2022  (R² = {r2:.4f})', fontsize=15, pad=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Revenue', fontsize=12)
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.5)
ax.annotate(
    f'MAE: {mae:,.0f}\nRMSE: {rmse:,.0f}',
    xy=(0.02, 0.88), xycoords='axes fraction',
    bbox=dict(boxstyle='round', fc='white', ec='gray', alpha=0.8)
)
plt.tight_layout()
plt.show()

### 5.2 · Feature Importance

In [ ]:
feature_imp = (
    pd.DataFrame({'Feature': FEATURES, 'Importance': model_rev.feature_importances_})
      .sort_values('Importance', ascending=False)
)

print('── FEATURE IMPORTANCE (Revenue model) ──')
print(feature_imp.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(x='Importance', y='Feature', data=feature_imp, palette='magma', ax=ax)
ax.set_title('Feature Importance — Revenue XGBoost', fontsize=14, pad=10)
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 6 · Recursive Forecasting — 2023 → 2024

> **Note:** `is_peak` includes March in both training and forecasting. Lag-7 is anchored to the same weekday one year prior to reduce long-horizon drift.

In [ ]:
future_dates = pd.date_range('2023-01-01', '2024-07-01', freq='D')
full_df      = pd.concat([df, pd.DataFrame({'Date': future_dates})], ignore_index=True)
origin_date  = sales['Date'].min()

for i in range(len(df), len(full_df)):
    d = full_df.loc[i, 'Date']

    # Calendar
    full_df.loc[i, 'trend_day']     = (d - origin_date).days
    full_df.loc[i, 'month']         = d.month
    full_df.loc[i, 'day_of_week']   = d.dayofweek
    full_df.loc[i, 'is_peak']       = int(d.month in [3, 4, 5, 6])
    full_df.loc[i, 'is_bottom']     = int(d.month in [10, 11, 12, 1])
    full_df.loc[i, 'is_wed_peak']   = int(d.dayofweek == 2)
    full_df.loc[i, 'is_sat_trough'] = int(d.dayofweek == 5)
    full_df.loc[i, 'dow_sin']       = np.sin(2 * np.pi * d.dayofweek / 7)

    # KPI projections
    for col in PROJ_COLS:
        pred_kpi = projection_models[col].predict([[full_df.loc[i, 'trend_day']]])[0]
        full_df.loc[i, f'proj_{col}'] = max(pred_kpi, df[col].quantile(0.1))

    # Lags
    full_df.loc[i, 'Revenue_lag1']      = full_df.loc[i-1,   'Revenue']
    full_df.loc[i, 'Revenue_lag7']      = full_df.loc[i-364, 'Revenue']
    full_df.loc[i, 'Revenue_roll7']     = full_df.loc[i-7:i-1, 'Revenue'].mean()
    full_df.loc[i, 'COGS_lag1']         = full_df.loc[i-1,   'COGS']
    full_df.loc[i, 'COGS_lag7']         = full_df.loc[i-364, 'COGS']
    full_df.loc[i, 'sessions_lag7']     = full_df.loc[i-364, 'sessions']
    full_df.loc[i, 'total_orders_lag7'] = full_df.loc[i-364, 'total_orders']

    # Predict
    full_df.loc[i, 'Revenue'] = max(0, np.expm1(model_rev.predict( full_df.loc[[i], FEATURES]))[0])
    full_df.loc[i, 'COGS']    = max(0, np.expm1(model_cogs.predict(full_df.loc[[i], FEATURES]))[0])

    # Carry-forward
    full_df.loc[i, 'sessions']     = full_df.loc[i-364, 'sessions']
    full_df.loc[i, 'total_orders'] = full_df.loc[i-364, 'total_orders']

print(f'✅ Forecast complete: {len(future_dates):,} days generated.')

## 7 · Export Submission

In [ ]:
submission = full_df.loc[full_df['Date'] >= '2023-01-01', ['Date', 'Revenue', 'COGS']].copy()
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
submission.to_csv('/kaggle/working/submission.csv', index=False)

print(f'✅ submission.csv saved — {len(submission):,} rows')
submission.head()